In [ ]:
!pip install -q "gensim>=4.3.3"
# numpy 互換エラーが出たら: ランタイム > セッションを再起動 して再実行


In [ ]:
# ===== knock77: GPU(CUDA)上での学習 =====
# 70〜76 の中身を1セルにまとめ、Colab の GPU(CUDA)で学習・評価する。
# 76 からの差分は device 周りの ★1〜★3 だけ。計算(マスク平均・BoW平均)は不変。
# 実行前に: ランタイム > ランタイムのタイプを変更 > ハードウェアアクセラレータ = GPU

import csv
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import gensim.downloader as api

PAD = "<PAD>"
LIMIT = 100000  # knock70 と同じ: 頻出上位10万語(語彙を揃えて 76 と比較可能に)
BATCH_SIZE = 64
LR = 0.01
EPOCHS = 10

# ★1 device 選択: Colab の GPU は CUDA。無ければ CPU にフォールバック。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 70: 埋め込み行列 ----
# 全3M語をロード(~2-3分, ~3.6GB RAM)し、頻度上位 LIMIT 語にスライス。
# index 上位=高頻度なのは bin の limit ロードと同じ。E[0] は PAD 用ゼロ行。
kv = api.load("word2vec-google-news-300")
vocab = kv.index_to_key[:LIMIT]
demb = kv.vector_size
E = np.zeros((len(vocab) + 1, demb), dtype=np.float32)
E[1:] = kv.vectors[:LIMIT]
word2id = {PAD: 0}
for i, w in enumerate(vocab):
    word2id[w] = i + 1


# ---- 71: データ読み込み ----
def load_sst2(path):
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)  # ヘッダ "sentence\tlabel"
        return [(s, l) for s, l in reader]


def text_to_ids(text, word2id):
    return [word2id[w] for w in text.split() if w in word2id]  # OOV は入れない


def build_dataset(data, word2id):
    ds = []
    for s, l in data:
        ids = text_to_ids(s, word2id)
        if not ids:  # 空列事例は捨てる
            continue
        ds.append(
            {
                "text": s,
                "label": torch.tensor([float(l)]),
                "input_ids": torch.tensor(ids, dtype=torch.long),
            }
        )
    return ds


# ---- 75: collate ----
def collate(batch):
    batch = sorted(batch, key=lambda ex: ex["input_ids"].size(0), reverse=True)
    seqs = [ex["input_ids"] for ex in batch]
    labels = [ex["label"] for ex in batch]
    input_ids = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "label": torch.stack(labels)}


# ---- 76: モデル(マスク平均 BoW → ロジ回帰) ----
class BoWClassifier(nn.Module):
    def __init__(self, E):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(E), freeze=True, padding_idx=0
        )
        self.fc = nn.Linear(self.emb.embedding_dim, 1)

    def forward(self, input_ids):
        vecs = self.emb(input_ids)  # (B, L, 300)
        mask = input_ids != 0
        lengths = mask.sum(dim=1, keepdim=True)  # (B, 1) 実トークン数
        summed = vecs.sum(dim=1)  # (B, 300) PAD は 0 で無害
        feat = summed / lengths  # (B, 300) 実長で割る = 平均
        return torch.sigmoid(self.fc(feat))  # (B, 1)


# ---- 73+77: 学習(GPU) ----
def train_model(model, train_data):
    loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
    )
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in loader:
            # ★2 各バッチを device へ(DataLoader は CPU テンソルを返すので毎回)
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            prob = model(ids)
            loss = criterion(prob, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * y.size(0)
        print(f"epoch {epoch}: loss = {total / len(train_data):.4f}")
    return model


# ---- 74+77: 評価(GPU) ----
def accuracy(model, data):
    loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            prob = model(ids)
            pred = (prob >= 0.5).float()
            correct += (pred == y).sum().item()
    return correct / len(data)


# ---- 実行 ----
# SST-2 の train.tsv / dev.tsv をアップロード(カレントに同名で保存される)。
from google.colab import files  # noqa: E402

print("train.tsv と dev.tsv を選択してアップロード:")
files.upload()

train = build_dataset(load_sst2("train.tsv"), word2id)
dev = build_dataset(load_sst2("dev.tsv"), word2id)
print("train:", len(train), "dev:", len(dev))

model = BoWClassifier(E).to(device)  # ★3 モデル(内部の埋め込み含む)を device へ

t0 = time.time()
train_model(model, train)
print("train time:", round(time.time() - t0, 1), "s")
print("dev accuracy:", accuracy(model, dev))
